In [1]:
!nvidia-smi

Wed May 27 07:54:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#Declaration

import os
import sys
import shutil
import logging
import csv
from pathlib import Path
from datetime import datetime

from ultralytics import YOLO

CONFIG = {
    "dataset_zip":    "./dataset.zip",
    "dataset_dir":    "./",
    "data_yaml":      "./data.yaml",
    "pretrained_model": "./model.pt",
    "base_model":     "yolov8n.pt",

    "epochs":         100,
    "imgsz":          640,
    "patience":       20,
    "batch":          16,
    "lr0":            0.01,
    "lrf":            0.01,
    "warmup_epochs":  3,
    "device":         0,

    "project":        "./runs/train",
    "run_name":       f"yolov8_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    "save_period":    10,
    "exist_ok":       False,
}

In [ ]:
#Utilities

def setup_logger(name: str = "yolov8_train") -> logging.Logger:
    """Configure a clean console + file logger."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    fmt = logging.Formatter(
        "[%(asctime)s] %(levelname)-8s %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    if logger.handlers:
        return logger

    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    try:
        base_path = Path(__file__).parent
    except NameError:
        base_path = Path.cwd()

    log_path = base_path / "train.log"

    fh = logging.FileHandler(log_path)
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    return logger

log = setup_logger()

def extract_dataset(zip_path: str, dest_dir: str) -> None:
    """Extract dataset archive if not already extracted."""
    zip_path = Path(zip_path)
    dest_dir = Path(dest_dir)

    if not zip_path.exists():
        raise FileNotFoundError(f"Dataset archive not found: {zip_path}")

    log.info(f"Extracting dataset: {zip_path} → {dest_dir}")
    shutil.unpack_archive(str(zip_path), str(dest_dir))
    log.info("Extraction complete.")


def resolve_model(pretrained_path: str, base_model: str) -> tuple[YOLO, str]:
    """
    Load model weights.

    Returns:
        (model, mode) where mode is 'fine-tune' or 'scratch'.
    """
    pretrained_path = Path(pretrained_path)

    if pretrained_path.exists():
        log.info(f"Found existing weights → fine-tuning: {pretrained_path}")
        return YOLO(str(pretrained_path)), "fine-tune"

    log.info(f"No existing weights found → training from scratch with: {base_model}")
    return YOLO(base_model), "scratch"


def validate_data_yaml(yaml_path: str) -> None:
    """Basic sanity-check that the data config file is present."""
    yaml_path = Path(yaml_path)
    if not yaml_path.exists():
        raise FileNotFoundError(f"data.yaml not found: {yaml_path}")
    log.info(f"Data config: {yaml_path}")

def build_train_args(config: dict, mode: str) -> dict:
    """
    Construct training kwargs from config.

    Fine-tuning uses a lower learning rate and fewer warm-up epochs
    to avoid overwriting learned features.
    """
    args = {
        "data":          config["data_yaml"],
        "epochs":        config["epochs"],
        "imgsz":         config["imgsz"],
        "patience":      config["patience"],
        "batch":         config["batch"],
        "device":        config["device"],
        "project":       config["project"],
        "name":          config["run_name"],
        "save_period":   config["save_period"],
        "exist_ok":      config["exist_ok"],
        "lrf":           config["lrf"],
    }

    if mode == "fine-tune":
        args["lr0"]           = config["lr0"] / 10
        args["warmup_epochs"] = 1
        log.info("Fine-tune mode: reduced lr0 and warm-up epochs.")
    else:
        args["lr0"]           = config["lr0"]
        args["warmup_epochs"] = config["warmup_epochs"]
        log.info("Scratch mode: standard learning rate schedule.")

    return args

def print_metrics_summary(metrics, best_weights: Path, run_dir: Path) -> None:
    """
    Print a formatted table of key detection metrics and save to CSV.

    YOLOv8 val() returns a Metrics object with attributes:
      box.p, box.r, box.map50, box.map  (overall)
      box.ap_class_index, box.ap50      (per-class mAP50)
    Note: YOLOv8 does not compute a standalone "accuracy" metric for
    object detection — mAP is the standard proxy used in the field.
    """
    box = metrics.box

    precision  = float(box.mp)
    recall     = float(box.mr)
    map50      = float(box.map50)
    map50_95   = float(box.map)

    DIVIDER = "─" * 50

    log.info(DIVIDER)
    log.info("  VALIDATION METRICS SUMMARY")
    log.info(DIVIDER)
    log.info(f"  {'Metric':<22} {'Value':>10}")
    log.info(DIVIDER)
    log.info(f"  {'Precision (mean)':<22} {precision:>10.4f}")
    log.info(f"  {'Recall (mean)':<22} {recall:>10.4f}")
    log.info(f"  {'mAP @ IoU=0.50':<22} {map50:>10.4f}")
    log.info(f"  {'mAP @ IoU=0.50:0.95':<22} {map50_95:>10.4f}")
    log.info(DIVIDER)
    log.info("  NOTE: 'Accuracy' is not a standard object-detection")
    log.info("  metric. mAP50 is the recognised equivalent.")
    log.info(DIVIDER)

    if hasattr(box, "ap_class_index") and hasattr(box, "ap50"):
        class_indices = box.ap_class_index
        class_ap50    = box.ap50

        if class_indices is not None and len(class_indices) > 0:
            names = getattr(metrics, "names", {}) or {}

            log.info("  PER-CLASS mAP50")
            log.info(DIVIDER)
            log.info(f"  {'Class':<22} {'mAP50':>10}")
            log.info(DIVIDER)

            per_class_rows = []
            for idx, ap in zip(class_indices, class_ap50):
                class_name = names.get(int(idx), f"class_{int(idx)}")
                log.info(f"  {class_name:<22} {float(ap):>10.4f}")
                per_class_rows.append({"class": class_name, "mAP50": round(float(ap), 4)})

            log.info(DIVIDER)

    log.info(f"  Best weights : {best_weights}")
    log.info(DIVIDER)

    csv_path = run_dir / "metrics_summary.csv"
    try:
        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["metric", "value"])
            writer.writerow(["precision",   round(precision, 4)])
            writer.writerow(["recall",      round(recall,    4)])
            writer.writerow(["mAP50",       round(map50,     4)])
            writer.writerow(["mAP50_95",    round(map50_95,  4)])

            if "per_class_rows" in dir() and per_class_rows:
                writer.writerow([])
                writer.writerow(["class", "mAP50"])
                for row in per_class_rows:
                    writer.writerow([row["class"], row["mAP50"]])

        log.info(f"  Metrics saved : {csv_path}")
    except Exception as e:
        log.warning(f"Could not save metrics CSV: {e}")

    log.info("=" * 60)

In [ ]:
#Main pipeline

def train(config: dict) -> None:
    """Full training pipeline."""
    log.info("=" * 60)
    log.info("YOLOv8 Training Pipeline Started")
    log.info("=" * 60)

    extract_dataset(config["dataset_zip"], config["dataset_dir"])
    validate_data_yaml(config["data_yaml"])

    model, mode = resolve_model(config["pretrained_model"], config["base_model"])
    log.info(f"Training mode : {mode.upper()}")
    log.info(f"Run name      : {config['run_name']}")

    train_args = build_train_args(config, mode)

    log.info("─" * 60)
    log.info("Training hyperparameters:")
    for k, v in train_args.items():
        log.info(f"  {k:<18}: {v}")
    log.info("─" * 60)

    results = model.train(**train_args)

    run_dir     = Path(config["project"]) / config["run_name"]
    best_weights = run_dir / "weights" / "best.pt"

    log.info("=" * 60)
    log.info("Training complete. Running validation on best weights...")
    log.info("=" * 60)

    if best_weights.exists():
        eval_model = YOLO(str(best_weights))
        metrics    = eval_model.val(data=config["data_yaml"], device=config["device"])
        print_metrics_summary(metrics, best_weights, run_dir)
    else:
        log.warning("best.pt not found — skipping evaluation summary.")

    return results

if __name__ == "__main__":
    try:
        train(CONFIG)
    except FileNotFoundError as e:
        log.error(f"Missing file: {e}")
        sys.exit(1)
    except Exception as e:
        log.exception(f"Training failed: {e}")
        sys.exit(1)